In [5]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

TRAIN_DIR = Path("../FullDataset/TrainingData")
TEST_DIR = Path("../FullDataset/TestData")
GROUND_TRUTH_DIR = TRAIN_DIR / "Ground Truth Package"

TRAIN_AUX_PATH = TRAIN_DIR / "AuxillaryTable.csv"
TRAIN_SPECTRA_PATH = TRAIN_DIR / "SpectralData.hdf5"
TARGET_PATH = GROUND_TRUTH_DIR / "FM_Parameter_Table.csv"

TEST_AUX_PATH = TEST_DIR / "AuxillaryTable.csv"
TEST_SPECTRA_PATH = TEST_DIR / "SpectralData.hdf5"

required_paths = [
    TRAIN_AUX_PATH,
    TRAIN_SPECTRA_PATH,
    TARGET_PATH,
    TEST_AUX_PATH,
    TEST_SPECTRA_PATH,
]

for path in required_paths:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

../FullDataset/TrainingData/AuxillaryTable.csv: FOUND
../FullDataset/TrainingData/SpectralData.hdf5: FOUND
../FullDataset/TrainingData/Ground Truth Package/FM_Parameter_Table.csv: FOUND
../FullDataset/TestData/AuxillaryTable.csv: FOUND
../FullDataset/TestData/SpectralData.hdf5: FOUND


Loads the training metadata, simulator targets and test
metadata into pandas DataFrames.

For each table, it reports:

- the number of rows and columns;
- the available column names;
- the number of duplicated planet identifiers;
- the total number of missing cells.

These checks establish the available metadata and target fields and provide an
initial integrity check. Unique planet identifiers are essential because the
future pipeline will align spectra, metadata and targets by `planet_ID`, rather
than assuming that separate files have the same row order.

In [7]:
train_aux = pd.read_csv(TRAIN_AUX_PATH)
targets = pd.read_csv(TARGET_PATH)
test_aux = pd.read_csv(TEST_AUX_PATH)

for name, table in {
    "training auxiliary": train_aux,
    "simulator targets": targets,
    "external test auxiliary": test_aux,
}.items():
    print(f"\n{name}")
    print("shape:", table.shape)
    print("columns:", table.columns.tolist())
    print("duplicate planet IDs:", table["planet_ID"].duplicated())
    print("missing values:", int(table.isna().sum().sum()))


training auxiliary
shape: (41423, 9)
columns: ['planet_ID', 'star_distance', 'star_mass_kg', 'star_radius_m', 'star_temperature', 'planet_mass_kg', 'planet_orbital_period', 'planet_distance', 'planet_surface_gravity']
duplicate planet IDs: 0        False
1        False
2        False
3        False
4        False
         ...  
41418    False
41419    False
41420    False
41421    False
41422    False
Name: planet_ID, Length: 41423, dtype: bool
missing values: 0

simulator targets
shape: (41423, 9)
columns: ['Unnamed: 0', 'planet_ID', 'planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
duplicate planet IDs: 0        False
1        False
2        False
3        False
4        False
         ...  
41418    False
41419    False
41420    False
41421    False
41422    False
Name: planet_ID, Length: 41423, dtype: bool
missing values: 0

external test auxiliary
shape: (685, 9)
columns: ['planet_ID', 'star_distance', 'star_mass_kg', 'star_radius_m', 'star_tem

### Inspect representative spectral HDF5 groups

The spectral files use HDF5, where each planet is stored in a separate group
named `Planet_<planet_ID>`. This cell opens each file in read only mode and
inspects one representative training planet and one representative test planet.

It reports the total number of planet groups, the selected group's attributes,
and the name, shape and data type of each spectral dataset. This confirms the
expected per-planet structure without loading the entire spectral dataset into
memory.

In [8]:
def inspect_spectral_file(
    path: Path,
    example_planet_id: str,
) -> None:
    group_name = f"Planet_{example_planet_id}"

    with h5py.File(path, "r") as handle:
        group = handle[group_name]

        print(f"\nFile: {path}")
        print("planet groups:", len(handle))
        print("example group:", group_name)
        print("group attributes:", dict(group.attrs))

        for dataset_name, dataset in group.items():
            print(
                dataset_name,
                "shape=", dataset.shape,
                "dtype=", dataset.dtype,
            )


inspect_spectral_file(TRAIN_SPECTRA_PATH, "train1")
inspect_spectral_file(TEST_SPECTRA_PATH, "public1")


File: ../FullDataset/TrainingData/SpectralData.hdf5
planet groups: 41423
example group: Planet_train1
group attributes: {'ID': 'train1'}
instrument_noise shape= (52,) dtype= float64
instrument_spectrum shape= (52,) dtype= float64
instrument_width shape= (52,) dtype= float64
instrument_wlgrid shape= (52,) dtype= float64

File: ../FullDataset/TestData/SpectralData.hdf5
planet groups: 685
example group: Planet_public1
group attributes: {'ID': 'public1'}
instrument_noise shape= (52,) dtype= float64
instrument_spectrum shape= (52,) dtype= float64
instrument_width shape= (52,) dtype= float64
instrument_wlgrid shape= (52,) dtype= float64


# NOTE:

The Ariel challenge test dataset is separate from the labelled data used for
the Track A train, validation, calibration and evaluation partitions. It
contains spectra and metadata but no simulator targets, so it cannot be used
to evaluate prediction accuracy or conformal coverage.

### Verify identifier alignment across source files

Spectra, auxiliary metadata and simulator targets are stored separately. They
must be joined using the explicit `planet_ID` field rather than their
row positions.

For spectral HDF5 files, the planet identifier is encoded in each root group
name as `Planet_<planet_ID>`. This check extracts those identifiers and compares
their sets with the corresponding CSV identifiers.

Matching identifier sets mean that each source contains the same planets, even
if its rows or groups appear in a different order. The future loader will still
perform an explicit identifier-based join; this audit does not make row-order
alignment acceptable.